# GibbsBASS Reproducibility Notebook

**Gibbs Posterior-Guided Bayesian Active Sparse Sampling for Uncertainty-Aware Compressive Speech Reconstruction**

This clean notebook reproduces the computational pipeline described in the GibbsBASS manuscript. It is intentionally organized to mirror the paper:

1. Dataset loading and preprocessing for LibriSpeech and Speech Commands v0.02.
2. Pilot measurement and sparse representation using an orthonormal DCT dictionary.
3. Gibbs posterior inference over the active sparse coefficient set.
4. Posterior-guided active sparse sampling.
5. Posterior-informed final reconstruction.
6. Baseline, ablation, uncertainty, runtime, statistical, and optional perceptual evaluation.

The implementation code is stored in `gibbsbass_reproducibility.py` so that the notebook remains readable and submission-ready.

## 1. Optional dependency installation

Run this cell in a fresh Google Colab or local environment. The optional packages `pesq` and `pystoi` are required only for PESQ, STOI, and ESTOI evaluation. They are not needed for the core reconstruction and uncertainty experiments.

In [ ]:
# Uncomment in a fresh environment if needed.
# %pip install numpy pandas scipy matplotlib librosa soundfile psutil pesq pystoi

## 2. Imports and manuscript-aligned configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from gibbsbass_reproducibility import (
    GibbsBASSConfig, METHODS, ABLATION_VARIANTS, MANUSCRIPT_REFERENCE_RESULTS,
    build_dataset, download_speech_commands_v002, run_full_benchmark, run_ablation,
    run_long_form_perceptual_evaluation, summarize_by_method, rank_methods,
    ablation_statistics, export_results
)

config = GibbsBASSConfig(output_dir="gibbsbass_outputs")
print(json.dumps({
    "target_sr": config.target_sr,
    "segment_length": config.segment_length,
    "max_segments_total": config.max_segments_total,
    "measurement_ratios": config.measurement_ratios,
    "seeds": config.seeds,
    "pilot_fraction": config.pilot_fraction,
    "active_set_size": config.active_set_size,
    "gibbs_iterations": config.gibbs_iterations,
    "burn_in": config.burn_in,
    "thinning": config.thinning
}, indent=2))

## 3. Dataset paths

The manuscript used two public datasets:

- **LibriSpeech ASR corpus**: https://www.openslr.org/12/
- **Google Speech Commands v0.02**: https://www.tensorflow.org/datasets/catalog/speech_commands and https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz

Set the extracted dataset directories below. In Google Colab, these may be Google Drive paths. If no paths are provided, the notebook can still run a small synthetic smoke test, but manuscript-level results require the public datasets.

In [ ]:
# Update these paths before the full manuscript run.
LIBRISPEECH_DIR = None  # e.g., "/content/data/LibriSpeech/dev-clean" or "/content/drive/MyDrive/LibriSpeech"
SPEECH_COMMANDS_DIR = None  # e.g., "/content/data/speech_commands_v0.02"

# Optional: automatically download Speech Commands v0.02.
DOWNLOAD_SPEECH_COMMANDS = False
if DOWNLOAD_SPEECH_COMMANDS and SPEECH_COMMANDS_DIR is None:
    SPEECH_COMMANDS_DIR = str(download_speech_commands_v002(Path(config.output_dir) / "data"))

print("LibriSpeech directory:", LIBRISPEECH_DIR)
print("Speech Commands directory:", SPEECH_COMMANDS_DIR)

## 4. Build the segment-level dataset

The paper uses controlled fixed-length 1,024-sample segments, resampled to 16 kHz, peak-normalized, and evaluated over a 40-segment benchmark. The dataset builder extracts active segments from both public datasets using the same preprocessing logic described in Section 3.3 of the manuscript.

In [ ]:
segments = build_dataset(
    librispeech_dir=LIBRISPEECH_DIR,
    speech_commands_dir=SPEECH_COMMANDS_DIR,
    config=config,
    synthetic_if_missing=True  # Keep True for a quick validation run when public data paths are absent.
)

segment_table = pd.DataFrame([{
    "Dataset": s["dataset"],
    "File": s["file"],
    "Start": s["start"],
    "Length": len(s["x"]),
    "Energy": float(np.mean(np.asarray(s["x"]) ** 2))
} for s in segments])

print("Number of segments:", len(segments))
display(segment_table.groupby("Dataset").agg(segments=("File", "count"), mean_energy=("Energy", "mean")))
segment_table.head()

## 5. Run the main benchmark

This cell evaluates the full manuscript method set: deterministic pursuit baselines, structured sensing baselines, FISTA baselines, Bayesian baselines, interpolation-guided sampling, and the full GibbsBASS framework.

For a quick environment check, leave `SMOKE_TEST_MODE=True`. For the full manuscript-consistent benchmark, set `SMOKE_TEST_MODE=False`, provide dataset paths above, and run all cells.

In [ ]:
SMOKE_TEST_MODE = True  # Set False for full manuscript-scale reproduction.

if SMOKE_TEST_MODE:
    # Lightweight validation while preserving the full implementation.
    smoke_config = GibbsBASSConfig(output_dir="gibbsbass_smoke_outputs")
    smoke_config.segment_length = 64
    smoke_config.max_segments_total = 1
    smoke_config.active_set_size = 8
    smoke_config.n_blocks = 4
    smoke_config.measurement_ratios = (0.30,)
    smoke_config.seeds = (42,)
    smoke_config.gibbs_iterations = 12
    smoke_config.burn_in = 4
    smoke_config.thinning = 2
    smoke_config.fista_max_iter = 20
    smoke_config.posterior_fista_max_iter = 30
    # Build a matching synthetic smoke dataset when the segment length differs.
    run_segments = build_dataset(None, None, smoke_config, synthetic_if_missing=True)
    run_methods = (
        "Random + OMP",
        "Random + FISTA",
        "Fixed Bayesian sparse recovery with Gibbs sampling",
        "Full proposed Gibbs posterior-guided Bayesian active sparse sampling framework (GibbsBASS)",
    )
    active_config = smoke_config
else:
    run_segments = segments
    run_methods = METHODS
    active_config = config

results_df = run_full_benchmark(run_segments, active_config, methods=run_methods)
print(results_df.shape)
results_df.head()

## 6. Main reconstruction summary and ranking

This reproduces the manuscript's Table 7-style method summary and overall ranking logic. The exact numerical values will match the manuscript when the same datasets, segment list, seeds, and runtime environment are used.

In [ ]:
ok_results = results_df[results_df.get("Status", "ok") == "ok"] if "Status" in results_df else results_df
method_summary = summarize_by_method(ok_results)
ranked_summary = rank_methods(method_summary)

cols = ["Method", "MSE", "RMSE", "MAE", "NMSE", "Relative error", "PRD", "SNR", "PSNR", "Overall rank score", "Overall rank"]
display(ranked_summary[[c for c in cols if c in ranked_summary.columns]])

## 7. Bayesian uncertainty and Gibbs diagnostics summary

This reproduces the manuscript's Bayesian reliability comparison by summarizing credible interval coverage, interval width, coverage error, uncertainty-error correlation, R-hat, ESS, MCSE, and autocorrelation.

In [ ]:
uncertainty_cols = [
    "Method", "95% CI coverage", "Mean CI width", "Coverage error",
    "Uncertainty-error correlation", "R-hat", "ESS", "MCSE", "Autocorrelation"
]
uncertainty_summary = ranked_summary[[c for c in uncertainty_cols if c in ranked_summary.columns]].dropna(subset=["95% CI coverage"], how="all")
display(uncertainty_summary)

## 8. Dataset-specific reconstruction summaries

This reproduces the manuscript's Table 11 and Table 12 logic by summarizing reconstruction behaviour separately for LibriSpeech and Speech Commands.

In [ ]:
dataset_summary = ok_results.groupby(["Dataset", "Method"], dropna=False).agg({
    "MSE": ["mean", "std"],
    "NMSE": ["mean", "std"],
    "PSNR": ["mean", "std"],
    "SI-SDR": ["mean", "std"]
}).reset_index()
dataset_summary.columns = [" ".join(c).strip() for c in dataset_summary.columns]
display(dataset_summary.head(20))

## 9. Module-level ablation study

This cell evaluates the full GibbsBASS design against the four module-level variants described in Section 3.8 and analyzed in Section 4.6 of the manuscript.

In [ ]:
if SMOKE_TEST_MODE:
    ablation_df = pd.DataFrame()
    print("Skipping full ablation in smoke-test mode. Set SMOKE_TEST_MODE=False for manuscript-scale ablation.")
else:
    ablation_df = run_ablation(run_segments, active_config, ratio=0.30)
    display(ablation_df.head())

## 10. Statistical significance analysis

This reproduces the Wilcoxon signed-rank, Holm-adjusted p-value, and Cliff's delta workflow used for the ablation significance analysis.

In [ ]:
if not ablation_df.empty:
    ablation_stats = ablation_statistics(ablation_df)
    display(ablation_stats)
else:
    ablation_stats = pd.DataFrame()
    print("Ablation statistics not generated in smoke-test mode.")

## 11. Optional long-form perceptual evaluation

This reproduces the manuscript's long-form PESQ/STOI/ESTOI evaluation when optional packages are installed. If `pesq` or `pystoi` is missing, the corresponding values will be returned as `NaN`.

In [ ]:
RUN_PERCEPTUAL_EVALUATION = False  # Set True when PESQ/STOI packages are installed and full data paths are available.

if RUN_PERCEPTUAL_EVALUATION:
    perceptual_df = run_long_form_perceptual_evaluation(run_segments, active_config, ratio=0.30, seed=42)
    display(perceptual_df)
else:
    perceptual_df = pd.DataFrame()
    print("Perceptual evaluation skipped. Set RUN_PERCEPTUAL_EVALUATION=True to compute PESQ/STOI/ESTOI.")

## 12. Export tables, figures, and configuration

This cell writes reproducibility outputs to disk: main results, ranked method summary, optional ablation results, statistical tests, configuration JSON, and basic figures.

In [ ]:
paths = export_results(results_df, ablation_df, active_config, active_config.output_dir)
if not perceptual_df.empty:
    perceptual_path = Path(active_config.output_dir) / "long_form_perceptual_metrics.csv"
    perceptual_df.to_csv(perceptual_path, index=False)
    paths["perceptual_metrics"] = str(perceptual_path)

print(json.dumps(paths, indent=2))

## 13. Manuscript consistency audit

The values below are the manuscript reference anchors. They are included only as an audit checklist. A full rerun should be compared against these values when the same public datasets, segment selection, seeds, and runtime configuration are used.

In [ ]:
pd.DataFrame([MANUSCRIPT_REFERENCE_RESULTS]).T.rename(columns={0: "Manuscript reference value"})

## 14. Notes for reviewers and re-users

- The notebook defaults to `SMOKE_TEST_MODE=True` to confirm that the environment and implementation are functional without requiring large downloads.
- For manuscript-scale reproduction, set `SMOKE_TEST_MODE=False`, provide the dataset paths in Section 3, and run all cells.
- LibriSpeech and Speech Commands are public datasets; processed subsets and experimental outputs can be regenerated using this notebook and `gibbsbass_reproducibility.py`.
- Minor numerical differences can occur across BLAS/LAPACK versions, random-number implementations, hardware, and optional dependency versions.